# Paper Visualizations — Refactored (CSV-based)
Todas las métricas se leen desde CSVs. No se evalúa ningún modelo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ------------------------------------------------------------------
# CSV_PATHS
# ------------------------------------------------------------------
BASE_MODELS_CSV   = Path("./base_models_results/holdout_test_metrics_all_models.csv") # base models trained on public dataset
CBAM_MODELS_CSV   = Path("./cbam_models_results/holdout_test_metrics_all_models.csv") # cbam models trained on public dataset
TL_BASE_MODELS_CSV = Path("./tl_base_models_results/domain_shift_tl_results.csv") # base models after transfer learning on local dataset
TL_CBAM_MODELS_CSV = Path("./tl_cbam_models_results/domain_shift_cbam_tl_results.csv") # cbam models after transfer learning on local dataset

# ------------------------------------------------------------------
# MODELS COLUMN (in holdout CSVs)
# the name of the column that contains the model names in the holdout CSVs
# ------------------------------------------------------------------
COL_MODEL   = "model"   

# ------------------------------------------------------------------
# CLASS METRICS COLUMNS (in holdout CSVs)
# assumedformat: "<class>_<metric>" 
# ------------------------------------------------------------------
CLASSES = [
    "chewing_insect",
    "abiotic_stress",
    "healthy",
    "mechanical_stress",
    "bacterial_spot",
    "mite_presence",
]

# metrics suffixes (e.g. F1, Recall, Precision) in the holdout CSVs
SUFFIX_F1        = "F1"         # e.g.  chewing_insect_F1
SUFFIX_RECALL    = "Recall"     # e.g.  chewing_insect_Recall
SUFFIX_PRECISION = "Precision"  # e.g.  chewing_insect_Precision

# ------------------------------------------------------------------
# TRANSFER LEARNING COLUMNS OF INTEREST (before / after FT, delta for the domain gap)
# ------------------------------------------------------------------
COL_TL_MODEL    = "model"       # model name column in the TL CSVs
COL_TL_F1_BEFORE = "F1_before"  
COL_TL_F1_AFTER  = "F1_after"   
COL_TL_DELTA_F1  = "delta_F1"   

# ------------------------------------------------------------------
# TARGET MODELS FOR COMPARISON (CBAM vs Base)
# ------------------------------------------------------------------
TARGET_MODELS = ["mobilenetv3large", "efficientnetb3", "resnet50"] # we can change this if the performance varies.

# ------------------------------------------------------------------
# OUTPUT DIR FOR THE VISUALIZATIONS
# ------------------------------------------------------------------
OUTPUT_DIR = Path("./visualizations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration routes: \n\n" )
print(f"   BASE_MODELS_CSV   : {BASE_MODELS_CSV}")
print(f"   CBAM_MODELS_CSV   : {CBAM_MODELS_CSV}")
print(f"   TL_BASE_MODELS_CSV: {TL_BASE_MODELS_CSV}")
print(f"   TL_CBAM_MODELS_CSV: {TL_CBAM_MODELS_CSV}")

## CBAM VS BASE MODELS
Class wise comparison (F1, Precision, Recall) for models: MobileNetV3Large, EfficientNetB3 y ResNet50.

In [ ]:
def load_holdout_csv(csv_path, model_col=COL_MODEL):
    """Loads a holdout CSV and normalizes the model names (strip + lower)."""
    if not csv_path.exists():
        raise FileNotFoundError(f"No se encontró el CSV: {csv_path}")
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[model_col])
    df[model_col] = df[model_col].str.strip().str.lower()
    return df


def extract_per_class_metrics(df, model_name, classes, model_col=COL_MODEL,
                               suf_f1=SUFFIX_F1, suf_rec=SUFFIX_RECALL, suf_prec=SUFFIX_PRECISION):
    """
    Extracts per-class metrics for a given model.
    Returns three arrays (f1, recall, precision), one for each metric,
    in the same order as `classes`.
    """
    row = df[df[model_col] == model_name.lower()]
    if row.empty:
        raise ValueError(f"Modelo '{model_name}' no encontrado en el CSV.")
    row = row.iloc[0] 

    f1_vals, rec_vals, prec_vals = [], [], []
    for cls in classes:
        f1_vals.append(float(row[f"{cls}_{suf_f1}"]))
        rec_vals.append(float(row[f"{cls}_{suf_rec}"]))
        prec_vals.append(float(row[f"{cls}_{suf_prec}"]))

    return np.array(f1_vals), np.array(rec_vals), np.array(prec_vals)


print("Loading holdout CSVs...")
df_base = load_holdout_csv(BASE_MODELS_CSV)
df_cbam = load_holdout_csv(CBAM_MODELS_CSV)

print(f"   Base models : {sorted(df_base[COL_MODEL].unique())}")
print(f"   CBAM models : {sorted(df_cbam[COL_MODEL].unique())}")

Gráfica para el paper:

In [ ]:
def plot_and_save_comparison(metrics_dict, model_name, classes, out_dir):
    """Plots and saves the comparison of Base vs CBAM metrics for a given model."""
    colors = ["#1f77b4", "#ff7f0e"] 
    f1_base   = metrics_dict["base"]["f1"]
    f1_cbam   = metrics_dict["cbam"]["f1"]
    prec_base = metrics_dict["base"]["precision"]
    prec_cbam = metrics_dict["cbam"]["precision"]
    rec_base  = metrics_dict["base"]["recall"]
    rec_cbam  = metrics_dict["cbam"]["recall"]

    display_classes = [c.replace("_", "\n") for c in classes]

    x = np.arange(len(classes))
    width = 0.35

    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    fig.subplots_adjust(wspace=0.3)

    for axis, (base_vals, cbam_vals), title in zip(
        ax,
        [(f1_base, f1_cbam), (prec_base, prec_cbam), (rec_base, rec_cbam)],
        ["F1 Score", "Precision", "Recall"]
    ):
        axis.bar(x - width/2, base_vals, width, label="Base", color=colors[0])
        axis.bar(x + width/2, cbam_vals, width, label="CBAM", color=colors[1])
        axis.set_xticks(x)
        axis.set_xticklabels(display_classes, rotation=45, ha="right", fontsize=8)
        axis.set_ylim(0, 1)
        axis.set_title(title)

    ax[2].legend(loc="upper right")
    sns.despine()
    plt.suptitle(model_name.capitalize(), fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()

    out_path = out_dir / f"cbam_comparison_{model_name}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Plot saved in: {out_path}")
    plt.show()
    plt.close()


def print_metrics_table(metrics_dict, model_name, classes):
    """Prints a table of metrics by class (Base vs CBAM)."""
    print(f"\n📊 Metrics by class for {model_name}:\n")
    for variant in ["base", "cbam"]:
        print(f"--- {variant.upper()} ---")
        f1        = metrics_dict[variant]["f1"]
        precision = metrics_dict[variant]["precision"]
        recall    = metrics_dict[variant]["recall"]
        for i, cls in enumerate(classes):
            print(f"  {cls:<22}  F1={f1[i]:.4f}  Prec={precision[i]:.4f}  Rec={recall[i]:.4f}")
        print(f"\n  📌 Macro  F1={np.mean(f1):.4f}  Prec={np.mean(precision):.4f}  Rec={np.mean(recall):.4f}\n")


output_comparison = OUTPUT_DIR / "cbam_comparisons"
output_comparison.mkdir(parents=True, exist_ok=True)

print("Creating comparison plots...\n")

for model_name in TARGET_MODELS:
    print(f"{'='*45}")
    print(f"🧠 Modelo: {model_name}")
    print(f"{'='*45}")

    try:
        f1_b, rec_b, prec_b = extract_per_class_metrics(df_base, model_name, CLASSES)
        f1_c, rec_c, prec_c = extract_per_class_metrics(df_cbam, model_name, CLASSES)
    except (ValueError, KeyError) as e:
        print(f"⚠️  Error al extraer métricas: {e}")
        continue

    metrics_dict = {
        "base": {"f1": f1_b, "recall": rec_b, "precision": prec_b},
        "cbam": {"f1": f1_c, "recall": rec_c, "precision": prec_c},
    }

    print_metrics_table(metrics_dict, model_name, CLASSES)
    plot_and_save_comparison(metrics_dict, model_name, CLASSES, output_comparison)

print("\n CBAM vs Base comparison process completed.")

## EVALUATING BEFORE VS AFTER FT
Comparing F1 macro on local dataset before and after fine-tuning models.

In [ ]:
def load_tl_csv(csv_path, model_col=COL_TL_MODEL):
    """Loads the TL CSV and normalizes the model name."""
    if not csv_path.exists():
        raise FileNotFoundError(f"No file found: {csv_path}")
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[model_col])
    df[model_col] = df[model_col].str.strip().str.lower()
    return df


print("Loading TL CSVs...")
df_tl_base = load_tl_csv(TL_BASE_MODELS_CSV)
df_tl_cbam = load_tl_csv(TL_CBAM_MODELS_CSV)

print(f"   TL Base models : {sorted(df_tl_base[COL_TL_MODEL].unique())}")
print(f"   TL CBAM models : {sorted(df_tl_cbam[COL_TL_MODEL].unique())}")

In [ ]:
def build_before_after_df(
    df_base, df_cbam,
    model_col=COL_TL_MODEL,
    col_before=COL_TL_F1_BEFORE,
    col_after=COL_TL_F1_AFTER,
):
    """
    Joins the Base and CBAM DataFrames, adds a `Model_Type` column
    and a readable name label for plotting.
    It considers the best strategy per model (the one that maximizes F1_after).
    """
    def _best_per_model(df, type_label, col_after=col_after):
        df = df.copy()
        df["Model_Type"] = type_label
        df = df.sort_values(col_after, ascending=False).groupby(model_col).head(1)
        return df

    df_b = _best_per_model(df_base, "Base")
    df_c = _best_per_model(df_cbam, "CBAM")
    combined = pd.concat([df_b, df_c], ignore_index=True)

    def _format_name(row):
        name = str(row[model_col]).capitalize()
        if row["Model_Type"] == "CBAM":
            return f"{name} (CBAM)"
        return name

    combined["Model_Plot"] = combined.apply(_format_name, axis=1)
    return combined


def plot_before_after_ft(
    df_combined,
    col_before=COL_TL_F1_BEFORE,
    col_after=COL_TL_F1_AFTER,
    out_dir=OUTPUT_DIR,
):
    """
    Generates the grouped bar chart for Before FT / After FT.
    """
    df_plot = pd.melt(
        df_combined,
        id_vars=["Model_Plot"],
        value_vars=[col_before, col_after],
        var_name="Condition",
        value_name="F1",
    )

    df_plot["Condition"] = df_plot["Condition"].replace({
        col_before: "Before FT",
        col_after:  "After FT",
    })

    order = (
        df_combined
        .sort_values(col_after, ascending=False)["Model_Plot"]
        .tolist()
    )

    sns.set_theme(style="whitegrid", font_scale=1.1)
    plt.figure(figsize=(16, 7))
    palette = sns.color_palette("viridis", 2)

    ax = sns.barplot(
        data=df_plot,
        x="Model_Plot", y="F1", hue="Condition",
        order=order,
        palette=palette, edgecolor="black", linewidth=1.1,
    )

    plt.xticks(rotation=45, ha="right")
    plt.ylabel("F1 Macro")
    plt.xlabel("")
    plt.title("F1 Macro — Before vs After Fine-Tuning", fontsize=14, fontweight="bold")
    plt.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.5)
    plt.ylim(0, 1.05)

    offset = 0.01
    for p in ax.patches:
        height = p.get_height()
        if abs(height) < 1e-6:
            continue
        y_pos = height + offset
        ax.annotate(
            f"{height:.3f}",
            (p.get_x() + p.get_width() / 2, y_pos),
            ha="center", va="bottom",
            fontsize=8, rotation=90,
        )

    plt.tight_layout()

    out_path = out_dir / "before_vs_after_ft.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved graph in: {out_path}")
    plt.show()
    plt.close()


print("Plotting Before vs After FT graph...")
df_combined = build_before_after_df(df_tl_base, df_tl_cbam)
plot_before_after_ft(df_combined, out_dir=OUTPUT_DIR)
print("\nComparison graph generated successfully.")

Delta F1 graph for paper:

In [ ]:
def plot_delta_f1(
    df_combined,
    col_before=COL_TL_F1_BEFORE,
    col_after=COL_TL_F1_AFTER,
    col_delta=COL_TL_DELTA_F1,
    out_dir=OUTPUT_DIR,
):
    """
    Generates a bar chart of the F1 delta (After - Before) by model.
    If the CSV already includes the delta column, it is used directly;
    otherwise, it is calculated.
    """
    df = df_combined.copy()

    if col_delta in df.columns:
        df["DeltaF1"] = df[col_delta]
        print(f"   Using column '{col_delta}' from CSV.")
    else:
        df["DeltaF1"] = df[col_after] - df[col_before]
        print(f"   Column '{col_delta}' not found → calculating {col_after} - {col_before}.")

    df = df.sort_values("DeltaF1", ascending=False)

    colors = ["#2ca02c" if d >= 0 else "#d62728" for d in df["DeltaF1"]]

    sns.set_theme(style="whitegrid", font_scale=1.1)
    plt.figure(figsize=(16, 6))

    ax = plt.bar(df["Model_Plot"], df["DeltaF1"], color=colors, edgecolor="black", linewidth=1.0)

    plt.axhline(0, color="black", linewidth=1.2, linestyle="--", alpha=0.7)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("ΔF1 (After − Before)")
    plt.xlabel("")
    plt.title("Improvement of F1 Macro after Fine-Tuning", fontsize=14, fontweight="bold")

    for bar, val in zip(ax, df["DeltaF1"]):
        offset = 0.003 if val >= 0 else -0.003
        va = "bottom" if val >= 0 else "top"
        plt.annotate(
            f"{val:+.3f}",
            (bar.get_x() + bar.get_width() / 2, val + offset),
            ha="center", va=va, fontsize=8, rotation=90,
        )

    plt.tight_layout()
    out_path = out_dir / "delta_f1_after_ft.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"✅ Saved graph in: {out_path}")
    plt.show()
    plt.close()


plot_delta_f1(df_combined, out_dir=OUTPUT_DIR)
print("\n🎉 Delta F1 graph generated!")